# Thalyann Olivo
June 8th, 2026

## Data Extraction

In [ ]:
import os
import glob
import json
import time
import requests
import pandas as pd

In [ ]:
"""
Extract ALL NYC 311 Service Requests (2020-present) and split them by borough.

Streams the full dataset from the Socrata Open Data API in a single ordered
pass using keyset pagination on the system :id field (fast at any depth, unlike
$offset), routes each row to a per-borough CSV, and checkpoints progress after
every page so the job can resume after an interruption.

"""

# ----------------------------------------------------------------- config
DOMAIN     = "data.cityofnewyork.us"
DATASET_ID = "erm2-nwe9"
APP_TOKEN  = "w2uZiRPbdj9PYUW6m8OPwJvup"
OUTPUT_DIR = "nyc311_by_borough"
PAGE_SIZE  = 50_000                      # rows per request; drop to 25_000 if you hit timeouts
DATE_START = "2020-01-01T00:00:00"       # inclusive
DATE_END   = "2026-05-31T23:59:59"       # inclusive

RESOURCE_URL = f"https://{DOMAIN}/resource/{DATASET_ID}.json"
CHECKPOINT   = os.path.join(OUTPUT_DIR, "_checkpoint.json")
HEADERS      = {"X-App-Token": APP_TOKEN} if "YOUR_APP" not in APP_TOKEN else {}


def soda_get(params, timeout=120):
    """GET with $ and : unencoded in the query string — required by Socrata SODA."""
    qs = "&".join(
        f"{urllib.parse.quote(str(k), safe='$:')}"
        f"={urllib.parse.quote(str(v), safe='$:,\'')}"
        for k, v in params.items()
    )
    r = requests.get(f"{RESOURCE_URL}?{qs}", headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return r.json()


def get_columns():
    """Infer column names from a single data row."""
    rows = soda_get({"$limit": 1, "$order": ":id"}, timeout=60)
    if not rows:
        raise RuntimeError("Dataset returned no rows")
    return [k for k in rows[0].keys() if not k.startswith(":")]


def borough_path(value):
    name = (value or "UNKNOWN").strip().upper()
    safe = "".join(ch if ch.isalnum() else "_" for ch in name) or "UNKNOWN"
    return os.path.join(OUTPUT_DIR, f"311_{safe}.csv")


def fetch_page(last_id, columns):
    """One page of rows ordered by :id, after the last :id we've seen."""
    where = f"created_date >= '{DATE_START}' AND created_date <= '{DATE_END}'"
    if last_id is not None:
        where += f" AND :id > '{last_id}'"
    params = {
        "$select": ":id, " + ", ".join(columns),
        "$where":  where,
        "$order":  ":id",
        "$limit":  PAGE_SIZE,
    }
    for attempt in range(6):
        try:
            return soda_get(params)
        except Exception as e:
            wait = 2 ** attempt
            print(f"  request failed ({e}); retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError("too many consecutive failures")


def write_page(rows, columns):
    """Append each row to its borough file; return the last :id in this page."""
    df = pd.DataFrame(rows)
    last_id = df[":id"].iloc[-1]                       # rows are sorted by :id asc
    df = df.drop(columns=[":id"]).reindex(columns=columns)
    for borough, group in df.groupby(df["borough"].fillna("UNKNOWN")):
        path = borough_path(borough)
        group.to_csv(path, mode="a", index=False,
                     header=not os.path.exists(path))
    return last_id


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    columns = get_columns()
    print(f"{len(columns)} columns; writing to ./{OUTPUT_DIR}/")

    last_id, total = None, 0
    if os.path.exists(CHECKPOINT):
        ck = json.load(open(CHECKPOINT))
        last_id, total = ck["last_id"], ck["total"]
        print(f"Resuming after :id {last_id} ({total:,} rows already written)")

    while True:
        rows = fetch_page(last_id, columns)
        if not rows:
            break
        last_id = write_page(rows, columns)
        total += len(rows)
        json.dump({"last_id": last_id, "total": total}, open(CHECKPOINT, "w"))
        print(f"{total:,} rows written  (last :id {last_id})")
        if len(rows) < PAGE_SIZE:        # last (partial) page
            break

    print(f"\nDone. {total:,} rows split across ./{OUTPUT_DIR}/")


if __name__ == "__main__":
    main()


28 columns; writing to ./nyc311_by_borough/
50,000 rows written  (last :id row-v9bh-je2k~p5a7)
100,000 rows written  (last :id row-ufvt~akgv~pspn)
150,000 rows written  (last :id row-ijfk-aeja~w5fk)
200,000 rows written  (last :id row-xdwr_xfqk-j5h9)
250,000 rows written  (last :id row-ykch-pmgs-2fk9)
300,000 rows written  (last :id row-szjc.ymnd~uu43)
350,000 rows written  (last :id row-3fcj_ycrf-zqwa)
400,000 rows written  (last :id row-fezx~zwqs~9sp5)
450,000 rows written  (last :id row-94if.i7qm_3pa6)
500,000 rows written  (last :id row-x67c.yexd.tbq4)
550,000 rows written  (last :id row-5pqv~ehjy~zzj7)
600,000 rows written  (last :id row-jfza~26px_zmti)
650,000 rows written  (last :id row-ynqr~ixmx.2d92)
700,000 rows written  (last :id row-wqmk.bx5t~vj78)
750,000 rows written  (last :id row-gst5_2993_mxzw)
800,000 rows written  (last :id row-5czj-2v6f.ngza)
850,000 rows written  (last :id row-i5q3.y9yp.mc6v)
900,000 rows written  (last :id row-u598~6mqs.8rhw)
950,000 rows written 

In [5]:
# Converting CSV to Parquet for faster loading in the notebook
CSV_DIR = "nyc311_by_borough"

csv_files = glob.glob(os.path.join(CSV_DIR, "311_*.csv"))
print(f"Found {len(csv_files)} borough CSV files")

for csv_path in csv_files:
    parquet_path = csv_path.replace(".csv", ".parquet")
    df = pd.read_csv(csv_path, low_memory=False)
    df.to_parquet(parquet_path, index=False)
    print(f"  {os.path.basename(csv_path)}  →  {os.path.basename(parquet_path)}  ({len(df):,} rows)")

print("Done.")

Found 7 borough CSV files
  311_QUEENS.csv  →  311_QUEENS.parquet  (5,110,647 rows)
  311_MANHATTAN.csv  →  311_MANHATTAN.parquet  (4,305,994 rows)
  311_STATEN_ISLAND.csv  →  311_STATEN_ISLAND.parquet  (898,716 rows)
  311_BRONX.csv  →  311_BRONX.parquet  (4,554,476 rows)
  311_BROOKLYN.csv  →  311_BROOKLYN.parquet  (6,381,501 rows)
  311_UNKNOWN.csv  →  311_UNKNOWN.parquet  (38,435 rows)
  311_UNSPECIFIED.csv  →  311_UNSPECIFIED.parquet  (39,296 rows)
Done.
